# AstralBridge — Kaggle DemoThis notebook runs the complete AstralBridge pipeline end to end on Kaggle:1. **Install** AstralBridge + dependencies2. **Configure** Gemma 4 API key from Kaggle Secrets3. **Download** the AION model (first run only, ~1.2 GB)4. **Generate** a survey adapter with Gemma 45. **Validate** the adapter through the repair loop6. **Run** real AION inference (redshift prediction)7. **Interpret** the prediction with Gemma 4**Requirements:** GPU accelerator (T4 or P100), Internet enabled.

## 1. Install AstralBridge

In [ ]:
# Clone the repository and install with torch + gemma extras!git clone https://github.com/Hasin-Raiyan/AstralBridge.git /kaggle/working/AstralBridge 2>/dev/null || echo 'Repo already cloned'%cd /kaggle/working/AstralBridge!pip install -e .[torch,gemma] -q!pip install astropy -qprint('Installation complete.')

## 2. Configure Gemma 4 API KeyAdd `GEMINI_API_KEY` as a Kaggle Secret (Add-ons → Secrets).Get a free key at https://aistudio.google.com/app/apikey

In [ ]:
import osfrom kaggle_secrets import UserSecretsClientsecrets = UserSecretsClient()os.environ["GEMINI_API_KEY"] = secrets.get_secret("GEMINI_API_KEY")print("GEMINI_API_KEY configured.")

## 3. Pre-download the AION ModelThe AION Base model (~1.2 GB) downloads automatically from HuggingFace onfirst use. This cell downloads it explicitly so you can see progress andverify it succeeded before running the full pipeline.

In [ ]:
import timeprint("Downloading AION model and codecs from HuggingFace...")print("This takes ~2-5 minutes on first run. Subsequent runs use the cache.")print()start = time.time()from aion.codecs import CodecManagerfrom aion.model import AIONimport torchdevice = "cuda" if torch.cuda.is_available() else "cpu"print(f"Device: {device}")# Load the model — this triggers the download if not cached.model = AION.from_pretrained("polymathic-ai/aion-base").to(device).eval()codec_manager = CodecManager(device=device)elapsed = time.time() - startprint(f"\nModel loaded in {elapsed:.0f}s.")print(f"Model parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.0f}M")print("Ready for inference.")

## 4. Prepare Example Survey DataAstralBridge is input-driven — you supply your own survey documentation,schema, and sample data. This demo uses a Legacy Survey DR9 example (thesurvey AION was trained on, so it is fully compatible).To run with your own data, replace these files with your survey's docs,schema, and sample observation.

In [ ]:
# Use the example data shipped with the repodocs_path = "/kaggle/working/AstralBridge/examples/survey_docs.txt"schema_path = "/kaggle/working/AstralBridge/examples/survey_schema.json"sample_path = "/kaggle/working/AstralBridge/examples/sample_observation.json"import jsonwith open(sample_path) as f:    sample = json.load(f)print(f"Survey: {sample['survey']}")print(f"Observation ID: {sample['observation_id']}")image = sample['image']print(f"Image shape: [{len(image)}, {len(image[0])}, {len(image[0][0])}]")print(f"Flux g/r/i/z: {sample['flux_g']}, {sample['flux_r']}, {sample['flux_i']}, {sample['flux_z']}")print(f"EBV: {sample['ebv']}")

## 5. Run the Full PipelineThis runs `astralbridge run` — the complete end-to-end flow:1. Gemma 4 generates a survey adapter2. Validation loop repairs until it passes3. Adapter produces a CanonicalObservation4. Bridge maps it to AION modality objects5. AION runs a real forward pass and decodes a redshift6. Gemma explains the prediction

In [ ]:
!astralbridge run \  --survey "Legacy Survey DR9" \  --docs {docs_path} \  --schema {schema_path} \  --sample {sample_path} \  --target redshift \  --output-dir /kaggle/working/out/ \  --device cuda \  --max-attempts 3

## 6. View Results

In [ ]:
import json, osout_dir = "/kaggle/working/out"print("=" * 55)print("GENERATED ADAPTER")print("=" * 55)with open(os.path.join(out_dir, "adapter.py")) as f:    print(f.read()[:2000])print()print("=" * 55)print("VALIDATION REPORT")print("=" * 55)with open(os.path.join(out_dir, "validation_report.json")) as f:    report = json.load(f)print(json.dumps(report, indent=2))print()print("=" * 55)print("PREDICTION (real AION output)")print("=" * 55)with open(os.path.join(out_dir, "prediction.json")) as f:    pred = json.load(f)print(json.dumps(pred, indent=2))print()print("=" * 55)print("GEMMA SCIENTIFIC INTERPRETATION")print("=" * 55)with open(os.path.join(out_dir, "interpretation.txt")) as f:    print(f.read())

## 7. SummaryThe pipeline is complete. Every step ran real code:- **Gemma 4** generated and validated the adapter (no hardcoded code)- **AION** ran a real forward pass with pretrained weights (no mocked inference)- **The prediction** is a real decoded redshift from the model's codebook- **Gemma 4** explained the real prediction (no placeholder text)### To run with your own survey:1. Replace the example files with your survey's documentation, schema, and sample data2. Run the same `astralbridge run` command3. AstralBridge will generate a new adapter specific to your survey format### Files produced:| File | Description ||------|-------------|| `out/adapter.py` | Generated and validated survey adapter || `out/validation_report.json` | Validation history (attempts, errors, manifest) || `out/prediction.json` | Real AION prediction (value, tokens, lineage) || `out/interpretation.txt` | Gemma's scientific explanation |